# Project Short — LTX Image-to-Video (Kaggle)

Notebook corrigé et simplifié pour Tesla T4.

**Ordre :** GPU → installation → chargement du modèle → image → paramètres → génération.

Pour une nouvelle vidéo dans la même session, modifier seulement la cellule **PARAMÈTRES DU PLAN** puis relancer **GÉNÉRATION**.

In [ ]:
# CELLULE 1 — VÉRIFIER LE GPU
import subprocess
import torch

subprocess.run(["nvidia-smi"])

print("\nPyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
print("Nombre de GPU :", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("Active un accélérateur GPU dans Kaggle avant de continuer.")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i} :", torch.cuda.get_device_name(i))
    print("VRAM :", round(props.total_memory / 1024**3, 2), "GB")

print("BF16 supporté :", torch.cuda.is_bf16_supported())

In [ ]:
# CELLULE 2 — INSTALLER DES VERSIONS COMPATIBLES
# Une seule fois après chaque nouveau démarrage complet de la session.
#
# IMPORTANT :
# diffusers 0.41.x demande une version récente de huggingface_hub,
# alors que transformers 4.51.3 exige huggingface_hub < 1.0.
# On utilise donc un ensemble de versions compatibles et testé ensemble.

import subprocess
import sys

packages = [
    "diffusers==0.39.0",
    "transformers==4.51.3",
    "tokenizers==0.21.4",
    "huggingface_hub==0.36.2",
    "accelerate>=1.2.0",
    "sentencepiece>=0.2.0",
    "imageio>=2.34",
    "imageio-ffmpeg>=0.5",
    "Pillow>=10",
    "safetensors>=0.8.0",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *packages],
    check=True,
)

print("✅ Dépendances compatibles installées.")
print("⚠️ Après cette cellule, redémarre la session Kaggle UNE FOIS si des paquets déjà chargés ont été remplacés.")


In [ ]:
# CELLULE 3 — CHARGER LTX AVEC GESTION MÉMOIRE
# Exécuter après la cellule 2 et, si nécessaire, après le redémarrage de session.

import gc
import torch
import diffusers
import transformers
import huggingface_hub

from diffusers import LTXImageToVideoPipeline

gc.collect()
torch.cuda.empty_cache()

print("diffusers :", diffusers.__version__)
print("transformers :", transformers.__version__)
print("huggingface_hub :", huggingface_hub.__version__)

MODEL_ID = "Lightricks/LTX-Video"

# Tesla T4 : FP16 est le choix le plus sûr.
DTYPE = torch.float16

print("Type numérique utilisé :", DTYPE)
print("Chargement du modèle...")

pipe = LTXImageToVideoPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)

# IMPORTANT : ne pas faire pipe.to("cuda").
# Les modules sont déplacés entre CPU et GPU au fur et à mesure.
pipe.enable_model_cpu_offload(gpu_id=0)
pipe.vae.enable_tiling()

try:
    pipe.enable_attention_slicing()
except Exception:
    pass

gc.collect()
torch.cuda.empty_cache()

print("✅ LTX chargé.")
print("pipe existe :", "pipe" in globals())
print("VRAM actuellement allouée :", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")


In [ ]:
# CELLULE 4 — TROUVER ET AFFICHER L'IMAGE
import glob
import os
from PIL import Image
import matplotlib.pyplot as plt

preferred_root = "/kaggle/input/datasets/pathemandiouba/lina-testt"
extensions = ("*.png", "*.jpg", "*.jpeg", "*.webp")

images = []
for ext in extensions:
    images.extend(glob.glob(os.path.join(preferred_root, "**", ext), recursive=True))

# Si le dossier change dans Kaggle, recherche automatique dans tous les inputs.
if not images:
    for ext in extensions:
        images.extend(glob.glob(os.path.join("/kaggle/input", "**", ext), recursive=True))

if not images:
    raise FileNotFoundError("Aucune image PNG/JPG/JPEG/WEBP trouvée dans /kaggle/input.")

print("Images trouvées :")
for i, path in enumerate(images):
    print(i, ":", path)

IMAGE_PATH = images[0]
print("\n✅ Image utilisée :", IMAGE_PATH)

preview = Image.open(IMAGE_PATH).convert("RGB")
plt.figure(figsize=(5, 8))
plt.imshow(preview)
plt.axis("off")
plt.show()

In [ ]:
# CELLULE 5 — PARAMÈTRES DU PLAN
# Pour les générations suivantes, c'est principalement CETTE cellule qu'il faut modifier.

PROMPT = """
Lina stands naturally and looks toward the camera. She blinks softly, smiles and gently waves with one hand. Her breathing is subtle and natural. Her long blonde hair moves only very slightly. Preserve the exact same character identity from the reference image: same face, same hairstyle, same pink shirt with heart, same light blue pants with flower details, same pink shoes, same proportions and same colors. Keep the background sharp and completely stable. The flowers and plants remain perfectly still. There is no wind and no camera movement. Smooth, natural children's animation with stable anatomy and consistent facial features.
""".strip()

NEGATIVE_PROMPT = """
worst quality, inconsistent motion, blurry, jittery, distorted, camera shake, zoom, pan, tilt, moving background, moving flowers, moving plants, wind, flicker, morphing, face deformation, body deformation, extra fingers, extra limbs, duplicated body parts, changing clothes, changing hairstyle
""".strip()

# Test léger recommandé pour Tesla T4.
WIDTH = 320
HEIGHT = 576
NUM_FRAMES = 33       # 8*n + 1 -> valeur valide
FPS = 16
NUM_INFERENCE_STEPS = 20
GUIDANCE_SCALE = 3.0
SEED = 42

OUTPUT_PATH = "/kaggle/working/lina_test.mp4"

print("✅ Paramètres prêts")
print("Image :", IMAGE_PATH)
print("Résolution :", f"{WIDTH}x{HEIGHT}")
print("Frames :", NUM_FRAMES)
print("FPS :", FPS)
print("Durée approximative :", round(NUM_FRAMES / FPS, 2), "secondes")
print("Sortie :", OUTPUT_PATH)

In [ ]:
# CELLULE 6 — GÉNÉRER LA VIDÉO
import gc
import os
import torch
from PIL import Image
from diffusers.utils import export_to_video
from IPython.display import Video, display

if "pipe" not in globals():
    raise RuntimeError("Le modèle n'est pas chargé. Exécute d'abord la cellule 3.")

if not os.path.isfile(IMAGE_PATH):
    raise FileNotFoundError(f"Image introuvable : {IMAGE_PATH}")

if WIDTH % 32 != 0 or HEIGHT % 32 != 0:
    raise ValueError("WIDTH et HEIGHT doivent être divisibles par 32.")

if (NUM_FRAMES - 1) % 8 != 0:
    raise ValueError("NUM_FRAMES doit respecter la forme 8*n + 1, par exemple 33, 49 ou 65.")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(0)

image = Image.open(IMAGE_PATH).convert("RGB")
image = image.resize((WIDTH, HEIGHT))

generator = torch.Generator(device="cpu").manual_seed(SEED)

print("🎬 Génération en cours...")
print("Ne ferme pas la session Kaggle pendant cette étape.")

result = pipe(
    image=image,
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    width=WIDTH,
    height=HEIGHT,
    num_frames=NUM_FRAMES,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
)

frames = result.frames[0]
export_to_video(frames, OUTPUT_PATH, fps=FPS)

peak = torch.cuda.max_memory_allocated(0) / 1024**3
print("✅ Vidéo terminée :", OUTPUT_PATH)
print("Pic de VRAM PyTorch :", round(peak, 2), "GB")

display(Video(OUTPUT_PATH, embed=True))

## Génération suivante

Tant que la session Kaggle reste active, **ne recharge pas le modèle**.

1. Modifie le prompt et/ou les paramètres dans la cellule 5.
2. Change `OUTPUT_PATH` pour ne pas écraser le plan précédent.
3. Relance uniquement la cellule 6.

Exemple :

`OUTPUT_PATH = "/kaggle/working/lina_turn_window.mp4"`

Si la session est totalement redémarrée, reprends depuis la cellule 1.